# La structure est nécessaire — C# — la conjecture d'Aaronson–Ambainis, prouvée et formalisée

**Série Complexity — accrétion 06d** · Public : **Recherche** · Hommage fondateur : **Scott Aaronson & Andris Ambainis** (*The need for structure in quantum speedups*, 2014) — et **David Turturean / Physical Superintelligence** (preprint 2026, formalisation Lean 4, dépôt `physical-superintelligence/aaronson-ambainis`)

Peut-on simuler tout algorithme quantique par un algorithme classique qui n'interroge l'entrée qu'en quelques points, sur la plupart des entrées ? En 2014, Aaronson et Ambainis ont conjecturé la pièce maîtresse : **tout polynôme borné de bas degré sur le cube — la trace de tout algorithme quantique à peu de requêtes — possède une variable influente**. Douze ans plus tard, un preprint de Physical Superintelligence (David Turturean) en donne une preuve complète, **formalisée en Lean 4 et vérifiée par deux noyaux indépendants**. Ce carnet enseigne l'énoncé, la chaîne de preuve en quatre maillons, et mesure sur des jouets ce que chaque maillon affirme : influences, degrés, certificats fractionnaires, algorithme C.

**Conventions de la série** : chaque fait est **Mesuré** (sortie de ce carnet) ou **Cité** (source nominative). L'honnêteté va plus loin ici : les nombres de ce carnet sont des **ombres numériques** de la preuve — elles vérifient les inégalités sur des instances finies, elles ne prouvent rien. La preuve est **Cité** : formalisée par les auteurs (`AaronsonAmbainis/Conjecture.lean`, pins v4.33.0), **non rejouée** dans nos lacs. Sources primaires archivées au gisement : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\2026 - Turturean (PSI) - A Proof of the Aaronson-Ambainis Conjecture (physical-superintelligence aaronson-ambainis 2a6ceae, preprint).pdf` et `2014 - Aaronson Ambainis - The Need for Structure in Quantum Speedups (Theory of Computing 10, arXiv 0911.0996v3, OA).pdf` (même dossier).

**Jumeau C#.** Cube, Walsh, influences, vecteurs quantiques et arbre C sont natifs ; les unitaires Haar utilisent MathNet et les LP HiGHS utilisent SciPy par Python.NET. Le Python versionné est la source primaire consultée pour le port : tous les propos sur le papier et Lean lui sont attribués ; aucune nouvelle lecture PDF ni aucun nouveau replay de preuve ne sont revendiqués. Les RNG diffèrent, et les résultats aléatoires ne sont pas comparés entre langages. [Source Python](Complexity-06d-AaronsonAmbainis-VariableInfluente-Python.ipynb).


**Navigation** : [<< Complexity-06c — Classical at heart](Complexity-06c-ClassicalAtHeart-Python.ipynb) | [Index de la série](README.md) | [Complexity-07 — Complexité de Kolmogorov bornée →](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb)

**Base** : [Complexity-06 — Simuler un circuit quantique classiquement](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) — ce carnet est l'accrétion **06d**. Prérequis ajoutés : polynômes multilinéaires sur le cube, transformée de Walsh–Hadamard, influences et variance, programmation linéaire (certificats fractionnaires).

## 1. L'énoncé — d'une conjecture de 2014 à un théorème formalisé

Aaronson et Ambainis (Cité : arXiv:0911.0996v3, Conjecture 6 ; *Theory of Computing* 10 (2014), Conjecture 1.7) écrivent :

> Let $p : \mathbb{R}^N \to \mathbb{R}$ be a polynomial of degree $d$. Suppose that $0 \le p(X) \le 1$ for all $X \in \{0,1\}^N$, and $\mathrm{E}[(p(X) - \mathrm{E}[p])^2] \ge \varepsilon$. Then there exists an $i$ such that $\mathrm{Inf}_i[p] \ge (\varepsilon/d)^{O(1)}$.

avec $\mathrm{Inf}_i[p] := \mathrm{E}[(p(X) - p(X^i))^2]$, où $X^i$ est $X$ avec le bit $i$ inversé. Le sens : un polynôme borné qui **varie** (variance $\ge \varepsilon$) possède une coordonnée dont l’influence est au moins la borne absolue polynomialement petite indiquée. Cela n’exige pas qu’elle soit strictement plus influente que toutes les autres : la moyenne, par exemple, peut répartir les influences également. Et c'est la bonne question pour le quantique : la probabilité d'acceptation d'un algorithme à $T$ requêtes est un polynôme de degré $\le 2T$ sur le cube (§4) ; si ce polynôme a une variable influente, un simulateur classique qui fixe cette variable reprend la main. La conjecture dit qu'un speedup quantique **exponentiel** exige de la structure, pas seulement du volume.

Le preprint PSI (Cité : Turturean, *A Proof of the Aaronson–Ambainis Conjecture*, 2026 — théorème 1.1 et corollaire AA01, rapportés comme lus par le Python dans sa copie archivée ; formalisation `AaronsonAmbainis/Conjecture.lean`, n'important que Mathlib) établit, en normalisation $\pm 1$ ($f : \{-1,1\}^n \to [-1,1]$, $\mathrm{Inf}_i(f) = \mathrm{E}[((f(x)-f(x^i))/2)^2]$) :

$$\mathrm{Inf}_j(f) \;\ge\; \frac{\sqrt{15}}{500\,\pi^4}\cdot\frac{\mathrm{Var}(f)^{5/2}}{d^6},$$

et dans la normalisation d'origine $[0,1]$ (déclaration `ExplicitBound` de `Conjecture.lean`) :

$$\mathrm{Inf}_i[p] \;\ge\; \frac{8\sqrt{15}}{125\,\pi^4}\cdot\frac{\varepsilon^{5/2}}{d^6} \;\ge\; \left(\frac{\varepsilon}{d}\right)^7 .$$

Le dépôt expose quatre théorèmes publics sans `sorry` ni axiome ajouté (`#print axioms` = `[propext, Classical.choice, Quot.sound]` sur chacun) : `aaronson_ambainis`, `aaronson_ambainis_explicit`, `aaronson_ambainis_pm` et `aaronson_ambainis_of_fbsDegreeBound` — ce dernier prend en **hypothèse externe** la borne d'Agarwal–Ben-David (arXiv:2410.19062, non prouvée dans le dépôt) qui remplace $\pi^2 d^2$ par $(\pi^2/2)d^2$ et multiplie la constante par 4 (`ExplicitBoundImproved` : $32\sqrt{15}/125\pi^4$). Vérification (Cité : README du dépôt, rapporté par le Python comme contrôlé sur son clone archivé au commit 2a6ceae) : audit statique sans `sorry`/`axiom`/`native_decide`, replay `leanchecker` sur chaque déclaration, second noyau indépendant lean4lean, et un « comparateur » qui rejoue un fichier-défi n'important que `Conjecture.lean` — avec contrôles négatifs : des solutions volontairement fausses y sont rejetées. Pins Lean/Mathlib **v4.33.0**, identiques à ceux de notre lac `sensitivity_lean` (§9).

In [1]:
#r "nuget: MathNet.Numerics,5.0.0"
#r "nuget: pythonnet,3.1.0"
#load "../../Config/PythonNetLoader.cs"
using System;
using System.Collections.Generic;
using System.Linq;
using System.Numerics;
using System.Globalization;
using System.Text.Json;
using Python.Runtime;
using MathNet.Numerics.LinearAlgebra.Complex;

// Stabiliser les décimales des résultats entre soumissions.
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;
double CAA = 8 * Math.Sqrt(15) / (125 * Math.Pow(Math.PI, 4));
double CPM = Math.Sqrt(15) / (500 * Math.Pow(Math.PI, 4));
if (Math.Abs(CAA - 32 * CPM) > 1e-15 || 512 * CAA < 1)
    throw new Exception("Constantes incorrectes");
Console.WriteLine($"c_AA={CAA:F6}, c_PM={CPM:E3}, ratio minimal={512 * CAA:F4}, hypothèse ABD : 4c_AA={4 * CAA:F6}");


Installed Packages MathNet.Numerics, 5.0.0 pythonnet, 3.1.0

c_AA=0.002545, c_PM=7.952E-005, ratio minimal=1.3029, hypothèse ABD : 4c_AA=0.010179


## Lecture du résultat — les constantes se répondent entre normalisations

Deux faits **Mesurés**. (1) $c_{AA} = 32\,c_{PM}$ exactement : le changement d'échelle $f = 2p-1$ conserve degré et influences mais multiplie la variance par 4, donc la constante gagne un facteur $4^{5/2} = 32$ — les deux énoncés sont le même théorème, et la valeur $c_{PM} \approx 7{,}95\times10^{-5}$ retrouvée ici est celle qu'annonce le papier. (2) Le rapport entre borne explicite et majorant imprimé vaut $c \cdot d/\varepsilon^{9/2}$ ; il est minimal aux bornes extrémales $\varepsilon = 1/4$ (le maximum possible : une variable de $[0,1]$ a une variance $\le 1/4$) et $d = 1$, où il vaut $512\,c \approx 1{,}30 > 1$ — la borne explicite implique donc le $(\varepsilon/d)^7$ de la conjecture imprimée **dans tout le domaine**, puisque le rapport croît dès qu'on s'écarte du pire cas.

## 2. La boîte à outils — cube, Fourier, influences, rangées

Un point du cube $\{0,1\}^n$ est codé par l'entier $x = \sum_k X_k 2^k$ ($0 \le x < 2^n$) ; $x^i = x \oplus 2^i$ est le voisin le long de la coordonnée $i$. En échelle $\pm 1$, $x_k = (-1)^{X_k}$ et $\chi_S(x) = \prod_{k \in S} x_k$ : les $\chi_S$ forment une base orthonormée, $f = \sum_S \hat f(S)\,\chi_S$, et la **transformée de Walsh–Hadamard** (papillons, $O(n\,2^n)$) calcule les $\hat f(S)$.

- **Degré** : le plus grand $|S|$ tel que $\hat f(S) \ne 0$ ;
- **Influences** : $\mathrm{Inf}_i[p] = \mathrm{E}[(p(X)-p(X^i))^2]$ (normalisation AA, $p \in [0,1]$) et $\mathrm{Inf}_i(f) = \mathrm{E}[((f(x)-f(x^i))/2)^2]$ (normalisation papier, $f = 2p-1$) — numériquement **égales** ; Parseval donne $\mathrm{Inf}_i(f) = \sum_{S \ni i} \hat f(S)^2$ ;
- **Incréments moyens** : $L_j(f) = \mathrm{E}\,|f(x) - f(x^j)|$ (le papier appelle $L_j/2$ l'influence $L^1$) ;
- **Rangée** au point $x$ (papier, déf. `row`) : des poids $a_i \ge 0$ tels que $\sum_{i \in S} a_i \ge |f(x) - f(x^{S})|$ pour **tout** $S \subseteq [n]$ non vide — un certificat fractionnaire : un budget réparti sur les coordonnées qui domine toutes les variations de $f$ autour de $x$. Son coût est $\lVert a \rVert_1$ ; le coût minimal en chaque point se calcule par programme linéaire (minimiser $\sum_i a_i$ sous les contraintes — solveur HiGHS de `scipy.optimize.linprog`).

Le bloc suivant définit l'outillage de tout le carnet et termine sur un contrôle croisé : influence directe contre identité de Parseval.

In [2]:
// Transformée de Walsh-Hadamard en place, normalisée par le nombre de sommets.
double[] Walsh(double[] valeurs)
{
    var a = (double[])valeurs.Clone();
    int n = a.Length;
    for (int h = 1; h < n; h *= 2)
        for (int b = 0; b < n; b += 2 * h)
            for (int j = 0; j < h; j++)
            {
                // Chaque papillon sépare les contributions paire et impaire du bit courant.
                double u = a[b + j], v = a[b + j + h];
                a[b + j] = u + v;
                a[b + j + h] = u - v;
            }

    // La normalisation donne les coefficients moyens sur le cube.
    for (int i = 0; i < n; i++)
        a[i] /= n;
    return a;
}

int Degre(double[] valeurs)
{
    var a = Walsh(valeurs);
    int d = 0;
    // Le poids du masque est le degré du caractère de Fourier correspondant.
    for (int s = 0; s < a.Length; s++)
        if (Math.Abs(a[s]) > 1e-9)
            d = Math.Max(d, BitOperations.PopCount((uint)s));
    return d;
}

double Variance(double[] a)
{
    double m = a.Average();
    return a.Average(x => (x - m) * (x - m));
}

double[] Influences(double[] a, int n, bool pm = false)
{
    var inf = new double[n];
    // Le passage de [0,1] à [-1,1] multiplie les différences au carré par quatre.
    double echelle = pm ? 0.25 : 1;
    for (int j = 0; j < n; j++)
        for (int x = 0; x < a.Length; x++)
        {
            // XOR inverse la coordonnée j pour comparer les deux sommets voisins.
            double d = a[x] - a[x ^ (1 << j)];
            inf[j] += echelle * d * d / a.Length;
        }

    return inf;
}

double[] Increments(double[] a, int n)
{
    var r = new double[n];
    for (int j = 0; j < n; j++)
        for (int x = 0; x < a.Length; x++)
            r[j] += Math.Abs(a[x] - a[x ^ (1 << j)]) / a.Length;
    return r;
}

double Tcheb(int d, double x) => Math.Cos(d * Math.Acos(Math.Clamp(x, -1, 1)));
Dictionary<string, double[]> Familles(int n)
{
    int N = 1 << n;
    var r = new Dictionary<string, double[]>();
    foreach (string nom in new[]
    {
        "dictateur",
        "ET_3",
        $"OU_{n}",
        "parite_2",
        "parite_4",
        "MAJ_3",
        "MAJ_5",
        "tribus_3x2",
        "adresse_2+4",
        "moyenne",
        "Tcheb_2(moy)",
        "Tcheb_3(moy)"
    }

    )
        r[nom] = new double[N];
    for (int x = 0; x < N; x++)
    {
        int k = BitOperations.PopCount((uint)x);
        double avg = 1.0 - 2.0 * k / n;
        r["dictateur"][x] = x & 1;
        r["ET_3"][x] = (x & 7) == 7 ? 1 : 0;
        r[$"OU_{n}"][x] = x != 0 ? 1 : 0;
        r["parite_2"][x] = BitOperations.PopCount((uint)(x & 3)) % 2;
        r["parite_4"][x] = BitOperations.PopCount((uint)(x & 15)) % 2;
        r["MAJ_3"][x] = BitOperations.PopCount((uint)(x & 7)) >= 2 ? 1 : 0;
        r["MAJ_5"][x] = BitOperations.PopCount((uint)(x & 31)) >= 3 ? 1 : 0;
        r["tribus_3x2"][x] = (x & 3) == 3 || (x & 12) == 12 || (x & 48) == 48 ? 1 : 0;
        // Les deux premiers bits sélectionnent un des quatre bits de données.
        int adresse = x & 3;
        r["adresse_2+4"][x] = (x >> (2 + adresse)) & 1;
        r["moyenne"][x] = (double)k / n;
        r["Tcheb_2(moy)"][x] = (1 + Tcheb(2, avg)) / 2;
        r["Tcheb_3(moy)"][x] = (1 + Tcheb(3, avg)) / 2;
    }

    return r;
}

// Vérifier Parseval indépendamment du calcul direct d'influence.
var controle = Familles(6)["tribus_3x2"];
var coeff = Walsh(controle);
var direct = Influences(controle, 6);
for (int j = 0; j < 6; j++)
{
    double somme = 0;
    for (int s = 0; s < coeff.Length; s++)
        if ((s & (1 << j)) != 0)
            somme += 4 * coeff[s] * coeff[s];
    if (Math.Abs(somme - direct[j]) > 1e-9)
        throw new Exception("Contrôle Parseval incorrect");
}

Console.WriteLine($"Parseval vérifié ; tribus degré={Degre(controle)}, somme influences={direct.Sum():F3}");
// SciPy/HiGHS est appelé pour les LP ; les autres calculs restent natifs C#.
PythonNetLoader.Initialize(typeof(PythonEngine).Assembly, "scipy");
PyObject lpScope;
using (Py.GIL())
{
    lpScope = Py.CreateScope();
    dynamic init = lpScope;
    init.Exec(@"import json, numpy as np, scipy
from scipy.optimize import linprog
version_scipy = scipy.__version__
def rangees(vals,n):
    v=np.asarray(vals,dtype=float)
    subsets=np.arange(1,1<<n)
    A=((subsets[:,None] >> np.arange(n)) & 1).astype(float)
    out=[]
    for x in range(1<<n):
        b=np.abs(v[x]-v[x ^ subsets])
        r=linprog(np.ones(n), A_ub=-A, b_ub=-b, bounds=[(0,None)]*n, method='highs')
        if not r.success or r.x is None or not np.isfinite(r.fun) or not np.all(np.isfinite(r.x)):
            raise RuntimeError('HiGHS a échoué au point %d: %s' % (x, r.message))
        if np.min(r.x)<-1e-8 or np.max(b-A@r.x)>1e-8:
            raise RuntimeError('Solution non réalisable au point %d' % x)
        out.append(float(r.fun))
    return json.dumps(out)");
    Console.WriteLine("SciPy/HiGHS version " + init.Get<string>("version_scipy"));
}

double[] Rangees(double[] valeurs, int n)
{
    if (valeurs.Length != (1 << n) || valeurs.Any(x => !double.IsFinite(x)))
        throw new ArgumentException("Valeurs du cube invalides");
    using (Py.GIL())
    {
        dynamic s = lpScope;
        string data = JsonSerializer.Serialize(valeurs);
        s.Exec($"rep = rangees(json.loads({JsonSerializer.Serialize(data)}), {n})");
        string raw = s.Get<string>("rep");
        double[] result = JsonSerializer.Deserialize<double[]>(raw);
        // Vérifier taille et finitude avant de réutiliser les optimums LP.
        if (result.Length != valeurs.Length || result.Any(x => !double.IsFinite(x) || x < -1e-8))
            throw new Exception("Résultat HiGHS invalide");
        return result;
    }
}

// Ginibre complexe, QR et correction des phases : vraies unitaires de Haar via MathNet.
double Normal(Random r)
{
    return Math.Sqrt(-2 * Math.Log(1 - r.NextDouble())) * Math.Cos(2 * Math.PI * r.NextDouble());
}

Complex[, ] Haar(int dim, Random r)
{
    var z = new Complex[dim, dim];
    for (int i = 0; i < dim; i++)
        for (int j = 0; j < dim; j++)
            z[i, j] = new Complex(Normal(r), Normal(r)) / Math.Sqrt(2);
    var qr = DenseMatrix.OfArray(z).QR();
    var u = new Complex[dim, dim];
    for (int j = 0; j < dim; j++)
    {
        var d = qr.R[j, j];
        var phase = d.Magnitude == 0 ? Complex.One : d / d.Magnitude;
        for (int i = 0; i < dim; i++)
            u[i, j] = qr.Q[i, j] * phase;
    }

    var m = DenseMatrix.OfArray(u);
    if ((m.ConjugateTranspose() * m - DenseMatrix.CreateIdentity(dim)).FrobeniusNorm() > 1e-10)
        throw new Exception("QR non unitaire");
    return u;
}

// L'oracle applique (-1)^x_q au secteur q ; le produit matriciel est calculé en C#.
double[] Acceptance(int n, List<Complex[, ]> unitaries, int[] acceptes)
{
    // Les secteurs de taille W distinguent les indices interrogés par l’oracle.
    int N = 1 << n, dim = unitaries[0].GetLength(0), W = dim / (n + 1);
    var output = new double[N];
    for (int x = 0; x < N; x++)
    {
        var state = new Complex[dim];
        state[0] = 1;
        for (int t = 0; t < unitaries.Count; t++)
        {
            // Une phase dépendant de x est appliquée entre deux unitaires.
            if (t > 0)
                for (int q = 1; q <= n; q++)
                    if ((x & (1 << (q - 1))) != 0)
                        for (int k = 0; k < W; k++)
                            state[q * W + k] = -state[q * W + k];
            var next = new Complex[dim];
            for (int i = 0; i < dim; i++)
                for (int j = 0; j < dim; j++)
                    next[i] += unitaries[t][i, j] * state[j];
            state = next;
        }

        // La somme des modules carrés projette sur les états acceptants.
        foreach (int i in acceptes)
            output[x] += state[i].Magnitude * state[i].Magnitude;
        if (!double.IsFinite(output[x]) || output[x] < -1e-10 || output[x] > 1 + 1e-10)
            throw new Exception("Probabilité invalide");
    }

    return output;
}

// Arbre exhaustif sur le cube : coût de calcul distinct du nombre moyen de requêtes.
(double[] estimation, int[] profondeur, List<double> influences) ArbreC(double[] vals, int n, double seuil)
{
    int N = 1 << n;
    var est = new double[N];
    var depth = new int[N];
    var infq = new List<double>();
    var pile = new Stack<(int mask, int value, int q)>();
    pile.Push((0, 0, 0));
    while (pile.Count > 0)
    {
        var(mask, value, q) = pile.Pop();
        // mask et value décrivent les coordonnées déjà fixées du sous-cube.
        var sub = Enumerable.Range(0, N).Where(x => (x & mask) == value).ToArray();
        var v = sub.Select(x => vals[x]).ToArray();
        // Une faible variance autorise une feuille estimée par la moyenne locale.
        if (Variance(v) <= seuil)
        {
            double avg = v.Average();
            foreach (int x in sub)
            {
                est[x] = avg;
                depth[x] = q;
            }

            continue;
        }

        // Parmi les variables libres, choisir celle dont l’influence locale est maximale.
        int choisi = -1;
        double max = -1;
        for (int i = 0; i < n; i++)
            if ((mask & (1 << i)) == 0)
            {
                double inf = sub.Average(x => Math.Pow(vals[x] - vals[x ^ (1 << i)], 2));
                if (inf > max)
                {
                    max = inf;
                    choisi = i;
                }
            }

        if (choisi < 0)
            throw new Exception("Sous-cube constant inattendu");
        infq.Add(max);
        // Les deux branches fixent le bit choisi à zéro puis à un.
        pile.Push((mask | (1 << choisi), value, q + 1));
        pile.Push((mask | (1 << choisi), value | (1 << choisi), q + 1));
    }

    return (est, depth, infq);
}


Parseval vérifié ; tribus degré=6, somme influences=1.688


SciPy/HiGHS version 1.16.3



warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'MathNet.Numerics' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'MathNet.Numerics' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you may need to supply runtime policy

warning CS1701: Assuming assembly reference 'System.Runtime.Numerics, Version=6.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' used by 'MathNet.Numerics' matches identity 'System.Runtime.Numerics, Version=10.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a' of 'System.Runtime.Numerics', you ma

## Lecture du résultat — le contrôle croisé passe

La sortie confirme (**Mesuré**) : l'influence directe $\mathrm{E}[(p-p^i)^2]$ égale la somme de Parseval $4\sum_{S \ni i} \hat p(S)^2$ à $10^{-9}$ près — l'outillage est cohérent avec la théorie qu'il mesure. Les `assert` intégrés aux tableaux qui suivent font partie du carnet : chaque famille de résultats porte sa propre vérification, et une inégalité n'est jamais seulement affichée, elle est aussi testée.

## 3. Familles jouets — la borne en nombres

On mesure la borne de `ExplicitBound` sur des familles classiques du cube à $n = 6$ : dictateur, ET, OU, parités, majorités, tribus (Ben-Or–Linial), fonction d'adressage, moyenne, Tchebychev de la moyenne. Deux résultats **Cités** y sont testés : le lemme 20 d'AA14 — $\sum_i \mathrm{Inf}_i[p] \le d$ — et l'invariance d'échelle $f = 2p-1$ (influences égales, variance ×4). La colonne **marge** est le rapport $\max_i \mathrm{Inf}_i / \text{borne}$ : la borne tient dès que marge $\ge 1$.

In [3]:
// Vérifier pour chaque famille les deux normalisations et la borne citée.
int n = 6;
var fam = Familles(n);
Console.WriteLine("famille | d | variance | max influence | somme influences | borne explicite | borne imprimée | marge");
foreach (var(nom, p)in fam)
{
    int d = Degre(p);
    double v = Variance(p);
    var inf = Influences(p, n);
    var f = p.Select(x => 2 * x - 1).ToArray();
    double bound = CAA * Math.Pow(v, 2.5) / Math.Pow(d, 6), printed = Math.Pow(v / d, 7);
    if (Math.Abs(Variance(f) - 4 * v) > 1e-12 || Influences(f, n, true).Zip(inf).Any(pair => Math.Abs(pair.First - pair.Second) > 1e-9) || inf.Sum() > d + 1e-9 || inf.Max() < bound - 1e-12)
        throw new Exception("Borne ou normalisation incorrecte");
    Console.WriteLine($"{nom, -16} {d, 2} {v:F4} {inf.Max():F4} {inf.Sum():F3} {bound:E3} {printed:E3} {inf.Max() / bound:E2}");
}


famille | d | variance | max influence | somme influences | borne explicite | borne imprimée | marge


dictateur         1 0.2500 1.0000 1.000 7.952E-005 6.104E-005 1.26E+004


ET_3              3 0.1094 0.2500 0.750 1.381E-008 8.562E-011 1.81E+007


OU_6              6 0.0154 0.0312 0.188 1.600E-012 7.275E-019 1.95E+010


parite_2          2 0.2500 1.0000 2.000 1.242E-006 4.768E-007 8.05E+005


parite_4          4 0.2500 1.0000 4.000 1.941E-008 3.725E-009 5.15E+007


... (3 outputs left out)

adresse_2+4       3 0.2500 0.5000 2.000 1.091E-007 2.791E-008 4.58E+006


moyenne           1 0.0417 0.0278 0.167 9.018E-007 2.180E-010 3.08E+004


Tcheb_2(moy)      2 0.0463 0.0617 0.370 1.834E-008 3.561E-012 3.37E+006


Tcheb_3(moy)      3 0.1240 0.1650 0.990 1.889E-008 2.058E-010 8.73E+006


## Lecture du résultat — une borne faite pour le régime asymptotique, pas pour les jouets

(**Mesuré**.) La borne tient partout (marge $\ge 1$, colonne de droite), mais avec un facteur $10^4$ (dictateur) à $10^{10}$ (OU_6) : sur des jouets, une coordonnée porte une influence macroscopique et la borne est très en dessous. Ce n'est pas un défaut mais l'objet même de la conjecture : elle vit dans le régime où $\max_i \mathrm{Inf}_i \to 0$ alors que la variance reste $\ge \varepsilon$ — la majorité (§5) est le laboratoire de ce régime.

Deux observations méritent l'œil. D'abord l'égalité $\sum \mathrm{Inf} = d$ du lemme 20, atteinte exactement par le dictateur ($1$), `parite_2` ($2$) et `parite_4` ($4$) : dans une parité, chaque bit retourne la valeur à coup sûr, et toutes les coordonnées pertinentes sont saturées. Ensuite la `moyenne`, de degré 1 comme le dictateur mais avec $\sum \mathrm{Inf} = 0{,}167$, très loin de $d$ : sa variance est éparpillée sur toutes les coordonnées, ce qui préfigure l'obstruction du §5.

## 4. D'où vient le polynôme — probabilités d'acceptation et degré $2T$

Le pont entre quantique et polynômes (Cité : AA14, lemme 21, attribué à Beals et al.) : la probabilité d'acceptation d'un algorithme quantique à $T$ requêtes est un polynôme multilinéaire de degré $\le 2T$ sur le cube — une requête n'est qu'une multiplication par les phases $(-1)^{X_k}$, de degré 1 par bit, et chaque bit n'est frappé qu'une fois par requête. C'est pourquoi la conjecture « parle » de polynômes bornés de bas degré : elle parle des algorithmes à peu de requêtes.

**Mesuré**, de deux façons. (a) *Circuits génériques* : des unitaires de Haar aléatoires ($n = 8$, $W = 2$ fils de travail par question) donnent des probabilités d'acceptation de degré exactement $2T$. (b) *Contrôle Deutsch* : le problème de promesse « bits égaux ou différents » se décide en **une** requête quantique — l'état $(e_{q=1} + e_{q=2})/\sqrt{2}$ préparé par une réflexion de Householder, une requête de phase, la réflexion inverse, mesure de l'état de départ. La probabilité d'acceptation $(1+(-1)^{x_1 \oplus x_2})/2$ est de degré $2 = 2T$, de variance $1/4$, et ses deux influences valent $1$.

In [4]:
// Réflexion de Householder qui prépare la superposition des deux questions.
int nd = 2, wd = 2, dim = (nd + 1) * wd;
var v = new double[dim];
v[2] = v[4] = 1 / Math.Sqrt(2);
var delta = new double[dim];
delta[0] = 1;
for (int i = 0; i < dim; i++)
    delta[i] -= v[i];
double norme = delta.Sum(x => x * x);
var house = new Complex[dim, dim];
for (int i = 0; i < dim; i++)
    for (int j = 0; j < dim; j++)
        house[i, j] = (i == j ? 1 : 0) - 2 * delta[i] * delta[j] / norme;
var deutsch = Acceptance(nd, new List<Complex[, ]> { house, house }, new[] { 0 });
var large = Acceptance(nd, new List<Complex[, ]> { house, house }, new[] { 0, 2, 4 });
if (!deutsch.Zip(new[] { 1.0, 0, 0, 1 }).All(z => Math.Abs(z.First - z.Second) < 1e-9) || large.Any(x => Math.Abs(x - 1) > 1e-9) || Degre(deutsch) != 2)
    throw new Exception("Contrôle Deutsch incorrect");
Console.WriteLine("Deutsch, une requête : " + string.Join(", ", deutsch.Select(x => x.ToString("F3"))) + " ; degré " + Degre(deutsch));
var deutschInf = Influences(deutsch, nd);
double deutschVar = Variance(deutsch);
if (Math.Abs(deutschVar - 0.25) > 1e-9 || deutschInf.Any(x => Math.Abs(x - 1) > 1e-9))
    throw new Exception("Variance ou influences de Deutsch incorrectes");
Console.WriteLine($"Variance={deutschVar:F3}, influences=({string.Join(", ", deutschInf.Select(x => x.ToString("F3")))})");
Console.WriteLine("Acceptation élargie : " + string.Join(", ", large.Select(x => x.ToString("F3"))));
double meilleur = 0;
for (int bit = 0; bit < 2; bit++)
    for (int sorties = 0; sorties < 4; sorties++)
    {
        int bonnes = 0;
        for (int x = 0; x < 4; x++)
            if (((sorties >> ((x >> bit) & 1)) & 1) == (((x & 1) == ((x >> 1) & 1)) ? 1 : 0))
                bonnes++;
        meilleur = Math.Max(meilleur, bonnes / 4.0);
    }

if (Math.Abs(meilleur - 0.5) > 1e-9)
    throw new Exception("Arbre classique incorrect");
Console.WriteLine($"Meilleur arbre classique à une requête, huit cas : {meilleur:F3}");
var rngH = new Random(2014);
int nh = 8, wh = 2, D = (nh + 1) * wh;
foreach (int T in new[]
{
    1,
    2,
    3
}

)
    for (int essai = 0; essai < 2; essai++)
    {
        var us = Enumerable.Range(0, T + 1).Select(_ => Haar(D, rngH)).ToList();
        var p = Acceptance(nh, us, Enumerable.Range(0, D).Where(i => i % wh == 0).ToArray());
        int degree = Degre(p);
        if (degree > 2 * T)
            throw new Exception("Degré > 2T");
        Console.WriteLine($"T={T} essai={essai} : degré={degree}, variance={Variance(p):E2}, maxInf={Influences(p, nh).Max():E2}");
    }


Deutsch, une requête : 1.000, 0.000, 0.000, 1.000 ; degré 2


Variance=0.250, influences=(1.000, 1.000)


Acceptation élargie : 1.000, 1.000, 1.000, 1.000


Meilleur arbre classique à une requête, huit cas : 0.500


T=1 essai=0 : degré=2, variance=1.49E-002, maxInf=4.11E-002


T=1 essai=1 : degré=2, variance=1.50E-002, maxInf=3.46E-002


T=2 essai=0 : degré=4, variance=1.11E-002, maxInf=2.03E-002


T=2 essai=1 : degré=4, variance=1.74E-002, maxInf=3.78E-002


T=3 essai=0 : degré=6, variance=1.41E-002, maxInf=3.27E-002


T=3 essai=1 : degré=6, variance=1.43E-002, maxInf=2.54E-002


## Lecture du résultat — le degré $2T$ est atteint, jamais dépassé ; et un piège de mesure

(**Mesuré**.) Sur circuits génériques, le degré mesuré égale $2T$ à chaque tirage : la borne du lemme 21 est **serrée** pour des algorithmes quelconques. Le contrôle Deutsch donne le même degré sur un exemple **exact** — exactitude quantique $1{,}0$ contre $0{,}5$ pour le meilleur arbre classique à une requête (les $8$ arbres énumérés plafonnent là) : l'écart quantique existe, et il vaut un facteur $2$ sur ce problème. La conjecture recadrée demande pourquoi il ne peut pas devenir **exponentiel** sans structure. La troisième sortie est l'avertissement : une « acceptation » trop large (mesurer tout le secteur de travail) rend $p \equiv 1$ — degré $0$, influence $0$. L'ensemble d'acceptation fait partie de l'algorithme ; une mesure paresseuse efface le signal que la requête venait de créer.

## 5. Pourquoi $\varepsilon$ et pourquoi $d$ — les deux obstructions

Une borne $\max_i \mathrm{Inf}_i \ge c\,\mathrm{Var}^\alpha / d^\beta$ ne peut économiser ni paramètre (Cité : papier, sous-sections « The exponent of the variance » et majorité). **Obstruction sur $\alpha$** : au degré 1, $\max_i \mathrm{Inf}_i \ge \mathrm{Var}^2$, avec égalité pour la moyenne $f = (x_1 + \dots + x_n)/n$ ($\mathrm{Var} = 1/n$, $\mathrm{Inf}_j = 1/n^2$) — donc $\alpha \ge 2$, sinon la moyenne violerait la borne aux grands $n$. **Obstruction sur $d$** : la majorité impaire garde $\mathrm{Var} = 1$ pendant que $\max \mathrm{Inf} = \binom{2k}{k}/4^k \sim 1/\sqrt{\pi k} \to 0$ avec $d = n$ — une borne indépendante de $d$, ou trop faible en $d$, serait fausse sur elle. PSI livre $\alpha = 5/2$ et $\beta = 6$ ; les deux tableaux mesurent ces deux étagères.

In [5]:
// La moyenne teste l’exposant en variance ; la majorité teste la dépendance en degré.
Console.WriteLine("Degré un : moyenne, égalité maxInf=Var²");
foreach (int m in new[]
{
    2,
    4,
    6,
    8,
    10
}

)
{
    var f = Enumerable.Range(0, 1 << m).Select(x => 1 - 2.0 * BitOperations.PopCount((uint)x) / m).ToArray();
    var inf = Influences(f, m, true);
    double v = Variance(f);
    if (Math.Abs(inf.Max() - v * v) > 1e-12 || Math.Abs(v - 1.0 / m) > 1e-12)
        throw new Exception("Obstruction moyenne");
    Console.WriteLine($"n={m}: Var={v:F4}, maxInf={inf.Max():F5}, Var²={v * v:F5}, degré={Degre(f)}");
}

Console.WriteLine("Majorité : influence maximale décroissante");
foreach (int m in new[]
{
    3,
    5,
    7,
    9,
    11,
    13
}

)
{
    var f = Enumerable.Range(0, 1 << m).Select(x => BitOperations.PopCount((uint)x) > m / 2 ? -1.0 : 1.0).ToArray();
    var inf = Influences(f, m, true);
    int k = (m - 1) / 2;
    double comb = 1;
    for (int j = 1; j <= k; j++)
        comb *= ((double)(k + j) / j) / 4;
    if (Math.Abs(inf.Max() - comb) > 1e-9 || Degre(f) != m)
        throw new Exception("Obstruction majorité");
    Console.WriteLine($"n={m}: Var={Variance(f):F3}, maxInf={inf.Max():F4}, centrale={comb:F4}, degré={Degre(f)}");
}


Degré un : moyenne, égalité maxInf=Var²


n=2: Var=0.5000, maxInf=0.25000, Var²=0.25000, degré=1


n=4: Var=0.2500, maxInf=0.06250, Var²=0.06250, degré=1


n=6: Var=0.1667, maxInf=0.02778, Var²=0.02778, degré=1


n=8: Var=0.1250, maxInf=0.01562, Var²=0.01562, degré=1


n=10: Var=0.1000, maxInf=0.01000, Var²=0.01000, degré=1


... (3 outputs left out)

n=7: Var=1.000, maxInf=0.3125, centrale=0.3125, degré=7


n=9: Var=1.000, maxInf=0.2734, centrale=0.2734, degré=9


n=11: Var=1.000, maxInf=0.2461, centrale=0.2461, degré=11


n=13: Var=1.000, maxInf=0.2256, centrale=0.2256, degré=13


## Lecture du résultat — $\alpha \ge 2$ forcé, $d$ incontournable

(**Mesuré**.) Premier tableau : pour la moyenne, $\mathrm{Var} = 1/n$ et $\max\mathrm{Inf} = 1/n^2 = \mathrm{Var}^2$ à chaque $n$. C'est le cas d'égalité de la borne de degré 1, et il force $\alpha \ge 2$ dans toute généralisation : le papier en fait l'argument dans ses remarques finales (exemple de degré 1, Cité) — $n^{-2} \ge c\,n^{-\alpha}$ pour tout $n$ exige $\alpha \ge 2$.

Second tableau : la majorité garde $\mathrm{Var} = 1$ tandis que $\max\mathrm{Inf}$ décroît comme $\binom{2k}{k}/4^k \approx 1/\sqrt{\pi k}$, avec un degré $d = n$. Une borne qui ne dépendrait que de la variance promettrait ici une influence constante, que la majorité dément : la dépendance en $d$ est **obligatoire** (c'est l'exemple de la majorité du papier, Cité). La borne PSI, elle, prédit $\mathrm{Inf} \ge c_{PM}/n^6$ — très en dessous de la vraie valeur $\approx 1/\sqrt{\pi k}$ : elle tient, au prix d'une forte perte dans ce régime.

D'où viennent les exposants $5/2$ et $6$ ? La remarque « The exponents » du papier (Cité) l'explique en trois temps : le moment d'ordre 3 (maillon 2) produit les exposants $3/2$ et $2$ du théorème d'incréments, $L_j \gtrsim \mathrm{Var}^{3/2}/K^2$ ; le moyennage (maillon 3) multiplie cette minoration par une quantité de l'ordre de $\mathrm{Var}/L$, d'où $\mathrm{Var}^{5/2}/(L\,K^2)$ ; enfin $K$ et $L$ sont de l'ordre de $d^2$ (maillon 4), d'où $d^6$ au dénominateur.

## 6. La chaîne de preuve en quatre maillons

(Cité : README « Proof at a glance » et sections 3–8 du papier ; chaque maillon y est un théorème, une proposition ou un corollaire formalisé dans le dépôt.) L'idée directrice : capturer les variations de $f$ par paires de points au moyen de certificats fractionnaires, intégrer sur les niveaux, moyenner sur les coordonnées — et payer chaque étape par la borne de rangée.

```mermaid
flowchart TD
    M1["Maillon 1 -- inegalite toutes-paires<br/>certificats fractionnaires + determinant"] -->|"integration sur les niveaux t"| M2["Maillon 2 -- moment d'ordre 3<br/>E max f-r 0 ^3 fois Pr f &lt;= r &lt;= 1.5 K^2 M"]
    M2 -->|"r = mediane, Cauchy-Schwarz"| M3["Maillon 3 -- moyennage<br/>L_j &gt;= Var^1.5 sur 6 K^2<br/>puis Inf_j &gt;= sqrt 15 sur 125 . Var^2.5 sur L K^2"]
    M4["Maillon 4 -- borne de rangee<br/>K = pi^2 d^2, Markov + Tchebychev + LP<br/>L = somme L_j &lt;= 4 d^2"] --> M3
    M3 -->|"K = pi^2 d^2 et L &lt;= 4 d^2"| T["Theoreme 1.1<br/>Inf_j &gt;= sqrt 15 sur 500 pi^4 . Var^2.5 sur d^6"]
```

Ce carnet ne rejoue **aucun** maillon — la preuve est celle du dépôt, vérifiée par ses deux noyaux. Il en **mesure les conclusions** (ombres) sur les familles du §3 :

1. le coût de rangée LP $K^*$ vérifie $K^* \le \pi^2 d^2$ (maillon 4) et $K^* \ge$ étendue de $f$ ;
2. la somme sensible ponctuelle $\sum_j |f(x) - f(x^j)|$ vérifie $\le 4d^2$ (d'où $L \le 4d^2$) ;
3. les $L_j$, le moment d'ordre 3 et les influences **mesurés** se placent du bon côté de leurs bornes (maillons 2–3) — chaque rapport doit être $\ge 1$ du côté des minorations, $\le 1$ du côté des majorations ;
4. l'hypothèse $0 < L \le K^*$ du théorème d'influence est satisfaite, et une coordonnée est **non légère** au sens de la remarque « The exponents » : $\mathrm{Inf}_j \ge \frac{2\,\mathrm{Var}}{5\,L}\,L_j$ (colonne `nonleger` : le rapport maximal sur $j$, qui doit être $\ge 1$).

In [6]:
// Chaque maillon compare une quantité calculée à sa borne avant affichage.
int nChain = 6;
var famillesChain = Familles(nChain);
Console.WriteLine("famille d K* π²d² sens 4d² L L<=K Lj/borne 3e/borne infl/borne nonleger");
foreach (var(nom, p)in famillesChain)
{
    var f = p.Select(x => 2 * x - 1).ToArray();
    int d = Degre(f);
    // K est le maximum des coûts LP sur tous les sommets de la famille.
    double K = Rangees(f, nChain).Max();
    double sens = 0;
    for (int x = 0; x < f.Length; x++)
    {
        double courant = 0;
        for (int j = 0; j < nChain; j++)
            courant += Math.Abs(f[x] - f[x ^ (1 << j)]);
        sens = Math.Max(sens, courant);
    }

    // L somme les incréments moyens ; M retient le plus grand.
    var lj = Increments(f, nChain);
    double L = lj.Sum(), v = Variance(f), M = lj.Max();
    var inf = Influences(f, nChain, true);
    double incr = M / (Math.Pow(v, 1.5) / (6 * K * K));
    double pire = 0;
    var seuils = Enumerable.Range(0, 201).Select(i => f.Min() + (f.Max() - f.Min()) * i / 200.0).Concat(f).Distinct();
    foreach (double r in seuils)
    {
        double a = f.Average(z => Math.Pow(Math.Max(z - r, 0), 3)) * f.Count(z => z <= r) / f.Length;
        double b = f.Average(z => Math.Pow(Math.Max(r - z, 0), 3)) * f.Count(z => z >= r) / f.Length;
        pire = Math.Max(pire, Math.Max(a, b));
    }

    double third = pire / (1.5 * K * K * M), ratioInf = inf.Max() / (Math.Sqrt(15) / 125 * Math.Pow(v, 2.5) / (L * K * K));
    double nonleger = Enumerable.Range(0, nChain).Where(j => lj[j] > 0).Max(j => inf[j] / ((2 * v / (5 * L)) * lj[j]));
    if (K > Math.PI * Math.PI * d * d + 1e-9 || K < f.Max() - f.Min() - 1e-9 || sens > 4 * d * d + 1e-9 || L > K + 1e-9 || incr < 1 - 1e-9 || third > 1 + 1e-9 || ratioInf < 1 - 1e-9 || nonleger < 1 - 1e-9)
        throw new Exception("Maillon violé pour " + nom);
    Console.WriteLine($"{nom, -16} {d, 2} {K, 7:F3} {Math.PI * Math.PI * d * d, 8:F2} {sens, 5:F2} {4 * d * d, 4} {L, 6:F3} oui {incr, 8:F1} {third, 7:F3} {ratioInf, 8:F1} {nonleger, 7:F2}");
}


famille d K* π²d² sens 4d² L L<=K Lj/borne 3e/borne infl/borne nonleger


dictateur         1   2.000     9.87  2.00    4  2.000 oui     48.0   0.167    258.2    2.50


ET_3              3   6.000    88.83  6.00   36  1.500 oui    373.2   0.032   3441.5    4.29


OU_6              6  12.000   355.31 12.00  144  0.375 oui   3538.6   0.009  58010.5    7.62


parite_2          2   4.000    39.48  4.00   16  4.000 oui    192.0   0.042   2065.6    5.00


parite_4          4   8.000   157.91  8.00   64  8.000 oui    768.0   0.010  16524.7   10.00


... (3 outputs left out)

adresse_2+4       3   6.000    88.83  6.00   36  4.000 oui    216.0   0.037   2323.8    5.00


moyenne           1   2.000     9.87  2.00    4  2.000 oui    117.6   0.036    632.5    2.50


Tcheb_2(moy)      2   6.667    39.48  6.67   16  2.500 oui   1394.3   0.007  15000.0    5.00


Tcheb_3(moy)      3  10.889    88.83 10.89   36  3.944 oui   1339.3   0.005  14378.8    4.99


## Lecture du résultat — la chaîne tient, et ses marges disent où elle perd

(**Mesuré**.) Les quatre ombres passent sur les douze familles.

1. **Rangées** : $K^* \le \pi^2 d^2$ partout, d'un facteur 5 environ (dictateur, moyenne : $2$ contre $9{,}87$) à près de 60 (tribus : $6$ contre $355{,}31$). Sur chaque jouet, $K^*$ coïncide avec la somme sensible ponctuelle maximale $\max_x \sum_j |f(x)-f(x^j)|$. L'inégalité $K^* \ge$ somme sensible est automatique — les contraintes à un seul bit l'imposent ; l'**égalité** mesurée dit que, sur ces jouets, le budget des flips simples domine déjà tous les flips multiples au point le plus coûteux.
2. **Somme sensible** : elle reste sous $4d^2$, donc $L \le 4d^2$ ; l'hypothèse $0 < L \le K^*$ du théorème d'influence tient partout, avec égalité $L = K^*$ pour le dictateur, les deux parités et la moyenne.
3. **Moments et influences** : le moment d'ordre 3 n'utilise qu'une fraction de son budget (rapport de $0{,}005$ pour Tcheb_3 à $0{,}167$ pour le dictateur) ; les $L_j$ dépassent leur minoration d'un facteur $48$ (dictateur) à environ $3{,}5\times10^3$ (OU_6), les influences d'un facteur $258$ à environ $5{,}8\times10^4$, aux deux mêmes extrémités.
4. **Coordonnée non légère** : elle existe partout (rapport de $2{,}50$ pour le dictateur et la moyenne à $10$ pour la parité 4).

Le dictateur est le jouet le plus serré à chaque étage (à égalité avec la moyenne sur les rangées, sur $L$ et sur la coordonnée non légère), OU_6 le plus lâche sur les deux minorations. Aucun jouet ne rend un maillon serré : le contenu mathématique est dans les instances limites, que la formalisation Lean couvre pour tout $n$ et que nos LP n'illustrent qu'à $n = 6$. Le §6 bis cherche le maillon serré là où le papier le situe.

## 6 bis. Le maillon 4 sous la loupe — la borne de rangée est-elle serrée ?

Le tableau précédent suggère que $\pi^2 d^2$ est une borne très lâche. Elle ne l'est pas toujours : le papier (Cité : sous-section « The constant in the row bound ») considère $f = T_d\big((x_1+\dots+x_n)/n\big)$ — Tchebychev de la moyenne, exactement les exemples de rigidité de Filmus–Hatami–Keller–Lifshitz — et montre qu'au point $\mathbf{1} = (1,\dots,1)$ toute rangée coûte au moins $n\,\big(1 - T_d(1-2/n)\big)$, quantité qui tend vers $2\,T'_d(1) = 2d^2$ quand $n \to \infty$. La constante admissible $c^*$ de la borne de rangée vit donc dans l'intervalle $[2, \pi^2/2]$ ($\pi^2/2$ via Agarwal–Ben-David). La LP confirme numériquement ce minorant dans les neuf cas testés, à la tolérance indiquée ; le second tableau applique ensuite la chaîne complète au cas le plus transparent, la moyenne ($d = 1$).

In [7]:
// Comparer les optimums LP aux formules de Tchebychev et de la moyenne.
foreach (int d in new[]
{
    1,
    2,
    3
}

)
    foreach (int m in new[]
    {
        4,
        6,
        8
    }

    )
    {
        var f = Enumerable.Range(0, 1 << m).Select(x => Tcheb(d, 1 - 2.0 * BitOperations.PopCount((uint)x) / m)).ToArray();
        double K = Rangees(f, m).Max(), bas = m * (1 - Tcheb(d, 1 - 2.0 / m));
        if (Math.Abs(K - bas) > 1e-4 || K > Math.PI * Math.PI * d * d + 1e-9)
            throw new Exception("Tchebychev incompatible");
        Console.WriteLine($"d={d}, n={m}: degré={Degre(f)}, K*={K:F3}, borne exacte={bas:F3}, 2d²={2 * d * d}, π²d²={Math.PI * Math.PI * d * d:F2}");
    }

foreach (int m in new[]
{
    4,
    6,
    8
}

)
{
    var f = Enumerable.Range(0, 1 << m).Select(x => 1 - 2.0 * BitOperations.PopCount((uint)x) / m).ToArray();
    double K = Rangees(f, m).Max(), L = Increments(f, m).Sum(), v = Variance(f);
    double bound = Math.Sqrt(15) / 125 * Math.Pow(v, 2.5) / (L * K * K);
    if (Math.Abs(K - 2) > 1e-9)
        throw new Exception("Rangée de la moyenne");
    Console.WriteLine($"n={m}: K*={K:F4}, L={L:F3}, borne={bound:E2}, influence={1.0 / (m * m):F4}, rapport={1.0 / (m * m) / bound:F0}");
}


d=1, n=4: degré=1, K*=2.000, borne exacte=2.000, 2d²=2, π²d²=9.87


d=1, n=6: degré=1, K*=2.000, borne exacte=2.000, 2d²=2, π²d²=9.87


d=1, n=8: degré=1, K*=2.000, borne exacte=2.000, 2d²=2, π²d²=9.87


d=2, n=4: degré=2, K*=6.000, borne exacte=6.000, 2d²=8, π²d²=39.48


d=2, n=6: degré=2, K*=6.667, borne exacte=6.667, 2d²=8, π²d²=39.48


d=2, n=8: degré=2, K*=7.000, borne exacte=7.000, 2d²=8, π²d²=39.48


... (2 outputs left out)

d=3, n=8: degré=3, K*=12.500, borne exacte=12.500, 2d²=18, π²d²=88.83


n=4: K*=2.0000, L=2.000, borne=1.21E-004, influence=0.0625, rapport=516


n=6: K*=2.0000, L=2.000, borne=4.39E-005, influence=0.0278, rapport=632


n=8: K*=2.0000, L=2.000, borne=2.14E-005, influence=0.0156, rapport=730


## Lecture du résultat — le point serré sur les cas testés, et le prix de l'uniformité

(**Mesuré**.) Premier tableau : pour $T_d$(moyenne), le coût LP coïncide avec $n(1 - T_d(1-2/n))$ à la tolérance numérique des neuf cas $(d,n)$ testés. Cette comparaison finie ne démontre pas l’identité pour tous les paramètres. Au degré 1, il vaut déjà $2 = 2d^2$ aux trois tailles testées : la constante $2$ est atteinte, pas seulement approchée. Aux degrés 2 et 3, il monte vers $2d^2$ quand $n$ croît ($6 \to 6{,}667 \to 7$ vers $8$ ; $8 \to 10{,}889 \to 12{,}5$ vers $18$). La borne de rangée ne peut donc pas descendre sous la constante $2$ ; le dépôt prouve $\pi^2$, la borne ABD donne $\pi^2/2$.

Second tableau : la moyenne a une rangée de coût $2$ aux tailles testées (les poids $(2/n, \dots, 2/n)$). La chaîne complète, de $K = 2$ et $L = 2$ jusqu'à l'influence, restitue pourtant une borne $516$, $632$ puis $730$ fois en dessous de l'influence vraie $1/n^2$ pour $n = 4, 6, 8$ — un écart qui **croît** avec $n$. Chaque maillon est franc, mais l'**accumulation** des pertes est le prix payé pour une borne valable pour toute fonction bornée de degré $d$.

## 7. Une inégalité bilatérale « en travers » — proposée première, prouvée troisième

Le dépôt démontre aussi (Cité : théorème « two-sided inequality » ; le README précise que la preuve du théorème principal **ne l'utilise pas**) : pour $g : Q_n \to \{0,1\}$ non constante et des rangées $\lambda^x$ quelconques,

$$4\;\mathrm{E}\!\left[\frac{\mathbf{1}_{g=1}}{\lVert\lambda^x\rVert}\right]\mathrm{E}\!\left[\frac{\mathbf{1}_{g=0}}{\lVert\lambda^x\rVert}\right] \;\le\; \max_j I_j(g),$$

avec **égalité** pour le dictateur et ses rangées $\lambda = (1,0,\dots,0)$ (exemple du papier). La chronologie (Cité : annexe « Timeline ») est parlante : proposée en premier des sept ingrédients, prouvée au troisième jour — en même temps que la première preuve complète, qui donnait alors l'exposant $22$ au lieu de $7$. Le tableau mesure le rapport avec les rangées LP minimales des familles booléennes du §3.

In [8]:
// La LP est calculée pour chaque sommet des familles booléennes.
foreach (var(nom, p)in Familles(6))
{
    if (p.Any(x => x != 0 && x != 1))
        continue;
    var cout = Rangees(p, 6);
    double un = 0, zero = 0;
    for (int i = 0; i < p.Length; i++)
    {
        if (p[i] == 1)
            un += 1 / cout[i] / p.Length;
        else
            zero += 1 / cout[i] / p.Length;
    }

    double lhs = 4 * un * zero, rhs = Influences(p, 6).Max();
    if (lhs > rhs + 1e-9)
        throw new Exception("Inégalité bilatérale");
    Console.WriteLine($"{nom, -16} min||l||={cout.Min():F3}, 4E+E-={lhs:F4}, maxInf={rhs:F4}, rapport={lhs / rhs:F3}");
}


dictateur        min||l||=1.000, 4E+E-=1.0000, maxInf=1.0000, rapport=1.000


ET_3             min||l||=1.000, 4E+E-=0.1458, maxInf=0.2500, rapport=0.583


OU_6             min||l||=1.000, 4E+E-=0.0103, maxInf=0.0312, rapport=0.328


parite_2         min||l||=2.000, 4E+E-=0.2500, maxInf=1.0000, rapport=0.250


parite_4         min||l||=4.000, 4E+E-=0.0625, maxInf=1.0000, rapport=0.062


MAJ_3            min||l||=1.500, 4E+E-=0.2934, maxInf=0.5000, rapport=0.587


MAJ_5            min||l||=1.667, 4E+E-=0.1617, maxInf=0.3750, rapport=0.431


tribus_3x2       min||l||=2.000, 4E+E-=0.1626, maxInf=0.2812, rapport=0.578


adresse_2+4      min||l||=3.000, 4E+E-=0.1111, maxInf=0.5000, rapport=0.222


## Lecture du résultat — l'égalité du dictateur, retrouvée numériquement

(**Mesuré**.) Dictateur : rapport $1{,}000$ — l'égalité prédite par l'exemple du papier, exactement retrouvée par la LP. Toutes les autres familles : rapport strictement $< 1$ — l'inégalité tient strictement, et sa hiérarchie est instructive (parité-4 au plus bas : la fonction la plus « délocalisée » est la plus loin du cas d'égalité). C'est la signature d'un théorème exact : son cas d'égalité est identifiable, et l'instrument tombe dessus — la même chose s'était produite pour la moyenne au §5 ($\mathrm{Inf} = \mathrm{Var}^2$ exactement).

## 8. La contrepartie algorithmique — l'algorithme C

La conjecture n'est pas qu'un comptage : AA14 en tire un **simulateur** (Cité : AA14, théorème 22). L'algorithme C, recevant $\varepsilon, \delta$, restreint le cube tant que la variance du polynôme restreint dépasse $\varepsilon^2\delta/2$ : à chaque pas il interroge une coordonnée d'influence suffisante et fixe sa valeur. À l'arrêt, Chebyshev donne $\Pr[|p_j - \mathrm{E}[p_j]| > \varepsilon] < \delta/2$ ; le comptage des influences ($\mathrm{E}[\text{itérations}] \le \mathrm{Inf}[p_0]/w$, troncature par Markov, union bound) borne l'échec total par $\delta$. Le corollaire 23 en dérive $D_{\varepsilon+\delta}(f) \le (Q_\varepsilon(f)/\delta)^{O(1)}$ : complexités classique et quantique sont polynomialement liées.

**Mesuré** : l'algorithme C en arbre de décision (profondeur = nombre de requêtes), sur des probabilités d'acceptation Haar du §4 ($n = 10, 12$ ; $T = 1, 2$ ; $\varepsilon = \delta = 0{,}1$). Cette implémentation énumère le cube et ne prouve donc pas un temps polynomial ; la borne citée concerne le nombre moyen de requêtes sur les entrées. Le tableau vérifie la garantie d'exactitude et le comptage du théorème ($\mathrm{E}[\text{profondeur}] \le \sum_i \mathrm{Inf}_i[p_0] / w_{\min}$, avec $w_{\min}$ la plus petite influence effectivement interrogée).

In [9]:
// Les arbres sont calculés sur les 2^n points ; les profondeurs comptent les requêtes.
var rngTree = new Random(7);
double eps = 0.1, delta = 0.1, seuil = eps * eps * delta / 2;
Console.WriteLine($"Seuil de variance : {seuil:F5}");
foreach (int m in new[]
{
    10,
    12
}

)
    foreach (int T in new[]
    {
        1,
        2
    }

    )
    {
        int W = 2, D = (m + 1) * W;
        var us = Enumerable.Range(0, T + 1).Select(_ => Haar(D, rngTree)).ToList();
        var p = Acceptance(m, us, Enumerable.Range(0, D).Where(i => i % W == 0).ToArray());
        var arbre = ArbreC(p, m, seuil);
        var inf = Influences(p, m);
        double som = inf.Sum();
        // La réussite est mesurée sur tous les sommets, pas sur un échantillon.
        double proba = Enumerable.Range(0, p.Length).Count(i => Math.Abs(arbre.estimation[i] - p[i]) <= eps) / (double)p.Length;
        double mean = arbre.profondeur.Average(), max = arbre.profondeur.Max();
        double wmin = arbre.influences.Count > 0 ? arbre.influences.Min() : double.PositiveInfinity;
        double bound = arbre.influences.Count > 0 ? som / wmin : 0;
        if (som > 2 * T + 1e-9 || proba < 1 - delta || mean > bound + 1e-9)
            throw new Exception("Contrôle arbre C");
        Console.WriteLine($"n={m}, T={T}: Var={Variance(p):E2}, maxInf={inf.Max():F4}, profondeur moy={mean:F2} (max {max}), Pr[erreur≤eps]={proba:F4}, wmin={wmin:E2}, sommeInf={som:F3}, borne moyenne={bound:F1}");
    }


Seuil de variance : 0.00050


n=10, T=1: Var=1.31E-002, maxInf=0.0344, profondeur moy=7.65 (max 10), Pr[erreur≤eps]=1.0000, wmin=1.14E-003, sommeInf=0.097, borne moyenne=85.1


n=10, T=2: Var=9.54E-003, maxInf=0.0158, profondeur moy=9.04 (max 10), Pr[erreur≤eps]=1.0000, wmin=1.28E-003, sommeInf=0.106, borne moyenne=83.4


n=12, T=1: Var=9.80E-003, maxInf=0.0176, profondeur moy=9.15 (max 12), Pr[erreur≤eps]=1.0000, wmin=8.73E-004, sommeInf=0.072, borne moyenne=82.2


n=12, T=2: Var=9.63E-003, maxInf=0.0169, profondeur moy=10.58 (max 12), Pr[erreur≤eps]=1.0000, wmin=1.04E-003, sommeInf=0.117, borne moyenne=112.4


## Lecture du résultat — la garantie tient ; l'asymptotique, elle, ne se voit pas à cette échelle

(**Mesuré**.) Pour les quatre bancs C# $(n,T)=(10,1),(10,2),(12,1),(12,2)$, la réussite observée est $1{,}0000$ ; les profondeurs moyennes valent $7{,}65$, $9{,}04$, $9{,}15$ et $10{,}58$, avec maxima $10$, $10$, $12$ et $12$. Les bornes de comptage associées valent $85{,}1$, $83{,}4$, $82{,}2$ et $112{,}4$ : elles dépassent $n$ et sont peu informatives à cette échelle. Les variances observées vont de $0{,}0095$ à $0{,}0131$ pour le seuil $0{,}0005$. Les unitaires proviennent des tirages Haar du port C# et diffèrent de celles du Python ; ces lectures ne sont pas une comparaison de performance entre langages.

(**Cité** : AA14, théorème 22, énoncé sous l'hypothèse de la conjecture, désormais démontrée.) Le gain est asymptotique : chaque requête fait baisser l'influence totale attendue d'au moins $w(\varepsilon^2\delta/2T)$, l'influence que la conjecture garantit au seuil, d'où un nombre attendu d'itérations au plus $\mathrm{Inf}[p_0]/w(\varepsilon^2\delta/2T) \le 2T/w(\varepsilon^2\delta/2T)$ — une quantité qui dépend de $T$, $\varepsilon$ et $\delta$, **pas de $n$**. C'est quand $n$ croît à $T$ fixé que la profondeur se détache de $n$ ; nos bancs à $n \le 12$ n'en montrent que la garantie d'exactitude. Notre $w_{\min}$ est la version mesurée de ce $w$ : la plus petite influence effectivement interrogée.

Le lien avec le [06](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) de la série passe par une distinction : le théorème 22 borne le nombre de **requêtes** à l'entrée, pas le **temps** de calcul. Notre algorithme C calcule $p$ sur les $2^n$ points pour restreindre sa variance — un coût de simulation que le théorème ne compte pas, et qui est précisément l'objet du 06. AA14 n'obtient le temps polynomial qu'au théorème 24, sous l'hypothèse supplémentaire $\mathsf{P} = \mathsf{P}^{\#\mathsf{P}}$. La conjecture dit qu'il suffit de lire quelques variables, celles qui portent la variance ; elle ne dit pas qu'il est facile de savoir lesquelles.

## 9. Contrepoint Huang — l'autre direction sur le même cube

Notre lac [`sensitivity_lean`](../../SymbolicAI/Lean/sensitivity_lean/README.md) (extraction de `Archive/Sensitivity.lean`, pins v4.33.0 — **identiques à ceux du dépôt PSI**) porte le théorème de Huang (2019) : le degré est dominé par la sensibilité, $\deg(f) \le s(f)^2$. Aaronson–Ambainis est l'autre sens : un **bas** degré force une **haute** influence. Lu ensemble sur une même fonction booléenne : Huang dit qu'une fonction peu sensible est simple, AA qu'une fonction de bas degré ne peut pas cacher sa variance. Le carnet [ANALYSE-06](../../SymbolicAI/Lean/Analyse/ANALYSE-06-Sensibilite-2026-Lean-Python.ipynb) (Cité, §4 « Ce que le lac ne contient pas ») rappelle les maillons non formalisés : l'équivalence de Gotsman–Linial et la borne de Nisan–Szegedy ($bs \le 2\deg^2$) ne sont pas dans le lac — le dossier complet sensibilité↔degré↔influence vit entre le lac et la littérature. Ombre **Mesurée** ci-dessous : $\deg \le s^2$ sur les familles booléennes du §3.


In [10]:
// Le même cube permet de comparer degré et sensibilité sur chaque famille booléenne.
foreach (var(nom, p)in Familles(6))
{
    if (p.Any(x => x != 0 && x != 1))
        continue;
    var sens = new int[p.Length];
    for (int x = 0; x < p.Length; x++)
        for (int i = 0; i < 6; i++)
            if (p[x] != p[x ^ (1 << i)])
                sens[x]++;
    int max = sens.Max(), d = Degre(p);
    if (d > max * max)
        throw new Exception("Contrôle Huang");
    Console.WriteLine($"{nom, -16} s={max}, s²={max * max}, degré={d}, sensibilité moyenne={sens.Average():F3}");
}


dictateur        s=1, s²=1, degré=1, sensibilité moyenne=1.000


ET_3             s=3, s²=9, degré=3, sensibilité moyenne=0.750


OU_6             s=6, s²=36, degré=6, sensibilité moyenne=0.188


parite_2         s=2, s²=4, degré=2, sensibilité moyenne=2.000


parite_4         s=4, s²=16, degré=4, sensibilité moyenne=4.000


MAJ_3            s=2, s²=4, degré=3, sensibilité moyenne=1.500


MAJ_5            s=3, s²=9, degré=5, sensibilité moyenne=1.875


tribus_3x2       s=3, s²=9, degré=6, sensibilité moyenne=1.688


adresse_2+4      s=3, s²=9, degré=3, sensibilité moyenne=2.000


## Lecture du résultat — deux théorèmes, deux directions, un seul cube

(**Mesuré**.) $\deg \le s^2$ partout, avec égalité pour le dictateur ($s = 1$, $\deg = 1$) — le même cas d'égalité que l'inégalité bilatérale du §7, et ce n'est pas un hasard : le dictateur est la fonction la plus concentrée du cube, celle où une seule coordonnée porte tout.

La colonne de sensibilité moyenne recoupe, famille par famille, la colonne $\sum \mathrm{Inf}$ du tableau du §3 ($1{,}000$ ; $0{,}750$ ; $0{,}188$ ; $2{,}000$ ; $4{,}000$ ; $1{,}500$ ; $1{,}875$ ; $1{,}688$ ; $2{,}000$). C'est une identité, pas une coïncidence : pour une fonction booléenne, $(p(x) - p(x^i))^2$ vaut $1$ exactement quand $x$ est sensible en $i$, donc $\mathrm{Inf}_i[p]$ est la probabilité d'être sensible en $i$, et $\sum_i \mathrm{Inf}_i[p]$ est la sensibilité moyenne. Les deux théorèmes parlent donc du même nombre : Huang borne le degré par la sensibilité **maximale**, AA minore l'influence **maximale** d'une fonction de bas degré, et la sensibilité **moyenne** — l'influence totale — est le pont entre les deux.

La comparaison des deux dépôts devient alors littérale : même Mathlib, même version d'outil, deux théorèmes miroirs. Les carnets [Lean-12 — Sensitivity Theorem](../../SymbolicAI/Lean/Lean-12-Sensitivity-Theorem.ipynb) et [ANALYSE-06b](../../SymbolicAI/Lean/Analyse/ANALYSE-06b-Sensibilite-2026-Lean.ipynb) développent le versant lac ; ce carnet referme le versant influence.


## 10. Provenance — une preuve produite par des modèles, vérifiée par des machines, provenance rapportée

L'aveu du papier (Cité : section de divulgation, rapportée par le Python comme lue dans sa copie archivée) est sans ambiguïté : les énoncés mathématiques ont été produits par des modèles d'IA (le papier nomme un modèle Anthropic non public) animant un harnais de recherche développé par David Turturean chez Physical Superintelligence — les instances cherchaient les idées existantes, en proposaient, les combinaient, se supervisaient ; l'auteur a cadré le problème, passé en revue et édité le brouillon rendu par une instance, et « is responsible for its correctness ». La preuve du théorème 1.1 et du corollaire AA a été formalisée en Lean et vérifiée par le noyau. L'annexe « Timeline » (Cité) donne la dynamique : jour 3, première preuve complète — avec l'exposant $22$ au lieu de $7$ ; puis $11$ (intégration sur les niveaux, vérifiée en Lean) ; puis $7$ (moyennage sur les coordonnées + $L \le 4d^2$, vérifié en Lean).

Ce carnet s'inscrit dans la même ligne que [07c — Le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) (372 preuves brutes) et [06c — Classical at heart](Complexity-06c-ClassicalAtHeart-Python.ipynb) (manuscrit unique en cours de vérification), avec un statut épistémique distinct et écrit : le Python rapporte une preuve **achevée et doublement vérifiée par machine** — le travail qui reste est humain : la lire, l'enseigner, la comparer. Statut **formalisée par les auteurs** (PSI, `Conjecture.lean`), **non rejouée** dans ce port ; ce que ce carnet apporte — l'enseignement, les ombres numériques, le croisement avec `sensitivity_lean` — est **nôtre** et **Mesuré**.

## Gap Mathlib

Au pin commun v4.33.0 (Cité : audit du carnet [ANALYSE-06](../../SymbolicAI/Lean/Analyse/ANALYSE-06-Sensibilite-2026-Lean-Python.ipynb), §« Lecture de l'audit »). Mathlib n'a ni sensibilité, ni influence, ni degré booléen : ces objets vivent dans la recherche, pas dans la librairie. Trois dépôts se partagent le terrain : `Archive/Sensitivity.lean` (théorème de Huang, extrait dans notre lac `sensitivity_lean`) ; le dépôt PSI `physical-superintelligence/aaronson-ambainis` (les quatre théorèmes AA, `import Mathlib` seul, zéro `sorry`, comparateur + lean4lean — **non vendu** chez nous à ce jour : son replay réclame ~7,4 Go en mono-thread et ~11 minutes d'après son README) ; et rien pour Gotsman–Linial ni Nisan–Szegedy. L'arbitrage de distillation tient en une ligne : non-distillable sur nos lacs sans port complet — un port partiel serait une régression face à `Conjecture.lean` — et la voie additive (carnet de lecture + comparaison : ce carnet) est celle livrée ici.


## Exercice 1 — changer d'échelle sans perdre les influences

Le tableau du §3 vérifie par `assert` que le passage $f = 2p - 1$ conserve les influences et multiplie la variance par 4. Écrire `VersPlusMoins(p)` qui retourne ce changement d'échelle, pour que la cellule affiche `True` sur ses deux contrôles.

In [11]:
double[] VersPlusMoins(double[] p)
{
    // TODO : convertir chaque valeur de [0,1] vers [-1,1].
    return null;
}

var ou4 = Enumerable.Range(0, 16).Select(x => x == 0 ? 0.0 : 1.0).ToArray();
var essai = VersPlusMoins(ou4);
var essaiAutre = VersPlusMoins(Familles(6)["moyenne"]);
if (essai is null && essaiAutre is null)
    Console.WriteLine("Exercice 1 à compléter");
else
{
    if (essai is null || essaiAutre is null)
        throw new Exception("Conversion partiellement remplie");
    void VerifierConversion(double[] p, int n, string nom)
    {
        var f = VersPlusMoins(p);
        if (f is null || f.Length != p.Length || f.Any(x => !double.IsFinite(x)))
            throw new Exception("Conversion absente ou invalide");
        for (int i = 0; i < p.Length; i++)
            if (Math.Abs(f[i] - (2 * p[i] - 1)) > 1e-12)
                throw new Exception("Conversion incorrecte");
        if (Math.Abs(Variance(f) - 4 * Variance(p)) > 1e-12 || Influences(f, n, true).Zip(Influences(p, n)).Any(z => Math.Abs(z.First - z.Second) > 1e-9))
            throw new Exception("Normalisation incorrecte");
        Console.WriteLine($"{nom}: variance ×4 et influences conservées");
    }

    VerifierConversion(ou4, 4, "OU_4");
    foreach (var nom in new[]
    {
        "dictateur",
        "moyenne",
        "tribus_3x2"
    }

    )
        VerifierConversion(Familles(6)[nom], 6, nom);
}


Exercice 1 à compléter


## Exercice 2 — le rapport minimal entre borne explicite et majorant imprimé

La cellule des constantes affiche $512\,c \approx 1{,}30$ au pire cas $\varepsilon = 1/4$, $d = 1$. Écrire `RatioMinimal(epsilon, d)` qui retourne le rapport $\big(c_{AA}\,\varepsilon^{5/2}/d^6\big)\big/(\varepsilon/d)^7$ — puis observer hors du pire cas qu'il ne fait que croître.

In [12]:
double? RatioMinimal(double epsilon, int d)
{
    // TODO : diviser la borne explicite par la borne imprimée.
    return null;
}

var premierRatio = RatioMinimal(0.25, 1);
var autreRatio = RatioMinimal(0.01, 4);
if (premierRatio is null && autreRatio is null)
    Console.WriteLine("Exercice 2 à compléter");
else
    foreach (var c in new (double eps, int d)[]
    {
        (0.25, 1),
        (0.01, 4),
        (0.1, 2)
    }

    )
    {
        var valeur = RatioMinimal(c.eps, c.d);
        double attendu = CAA * c.d / Math.Pow(c.eps, 4.5);
        if (valeur is null || !double.IsFinite(valeur.Value) || Math.Abs(valeur.Value - attendu) > 1e-9 * Math.Max(1, attendu))
            throw new Exception("Rapport incorrect");
        Console.WriteLine($"eps={c.eps}, d={c.d} : rapport {valeur.Value:G6}");
    }


Exercice 2 à compléter


## Exercice 3 — la rangée du OU au point $0^n$, à la main

Le §6 calcule les rangées par LP. Pour $f = 2\,\mathrm{OU}_n - 1$ au point $0^n$, la rangée minimale se détermine **sans LP** : depuis $0^n$, tout flip non vide fait passer $f$ de $-1$ à $+1$. Écrire `RangeeOuZero(n)` qui retourne ce coût minimal exact ; la cellule le confronte à la LP au point $0$ et au degré de $\mathrm{OU}_n$.

In [13]:
double? RangeeOuZero(int n)
{
    // TODO : utiliser les contraintes des singletons, puis sommer la rangée minimale.
    return null;
}

var premierCout = RangeeOuZero(6);
var autreCout = RangeeOuZero(2);
if (premierCout is null && autreCout is null)
    Console.WriteLine("Exercice 3 à compléter");
else
    foreach (int n in new[]
    {
        2,
        4,
        6
    }

    )
    {
        var f = Enumerable.Range(0, 1 << n).Select(x => x == 0 ? -1.0 : 1.0).ToArray();
        double attendu = Rangees(f, n)[0];
        var trouve = RangeeOuZero(n);
        if (trouve is null || !double.IsFinite(trouve.Value) || Math.Abs(trouve.Value - attendu) > 1e-7)
            throw new Exception("Coût de rangée incorrect");
        Console.WriteLine($"n={n}: coût proposé={trouve.Value:F3}, HiGHS={attendu:F3}, degré OU={Degre(f)}");
    }


Exercice 3 à compléter


## À retenir

Le [06](Complexity-06-Simuler-Circuit-Quantique-Classiquement-Python.ipynb) demandait le coût de *simuler* un circuit ; le [06b](Complexity-06b-AaronsonGottesman-Dequantification-Python.ipynb) exécutait le cas Clifford ; le [06c](Complexity-06c-ClassicalAtHeart-Python.ipynb) demandait si un oracle classique peut *implémenter* une opération quantique. Ce 06d referme la boucle par l'énoncé qui portait toute la série : **la structure est nécessaire** — un polynôme borné de degré $d$ et de variance $\varepsilon$ a une variable d'influence $\ge (8\sqrt{15}/125\pi^4)\,\varepsilon^{5/2}/d^6 \ge (\varepsilon/d)^7$, donc un algorithme quantique à $T$ requêtes ne bat le classique que polynomialement sur la plupart des entrées (corollaire 23 d'AA14, via l'algorithme C du §8). La preuve est de 2026, produite par des modèles, formalisée en Lean, vérifiée par deux noyaux — et les expériences de ce carnet en illustrent des conséquences finies : degré $2T$ sur des circuits simulés, coûts de rangée par LP, égalités du dictateur retrouvées, garanties de l'algorithme C vérifiées. Le maillon suivant de la chaîne sensibilité↔influence qui manque à nos lacs — Gotsman–Linial, Nisan–Szegedy — est exactement le dossier qu'ouvre le croisement [`sensitivity_lean`](../../SymbolicAI/Lean/sensitivity_lean/README.md) ↔ PSI.


## Vérifiez votre compréhension

1. Le §4 mesure des probabilités d'acceptation de degré exactement $2T$, et le §2 l'identité $\mathrm{Inf}_i(f) = \sum_{S \ni i} \hat f(S)^2$. Pourquoi la conjecture AA est-elle un énoncé sur des polynômes, et non sur des circuits ?
2. La borne explicite est $c\,\varepsilon^{5/2}/d^6$ avec $c \approx 2{,}5\times10^{-3}$. Pourquoi le rapport $512\,c > 1$ calculé à $\varepsilon = 1/4$, $d = 1$ suffit-il à impliquer le majorant imprimé $(\varepsilon/d)^7$ **dans tout le domaine** ?
3. Le §9 oppose Huang ($\deg \le s^2$) et AA (bas degré ⇒ haute influence). Donner, pour la majorité impaire, ses $s$, $\deg$, $\max\mathrm{Inf}$ et $\mathrm{Var}$ — et expliquer pourquoi elle ne contredit ni l'un ni l'autre.

<details><summary>Réponses</summary>

1. Par le lemme 21 (Beals et al.) : la probabilité d'acceptation d'un algorithme à $T$ requêtes **est** un polynôme multilinéaire de degré $\le 2T$ à valeurs dans $[0,1]$ — dans l'énoncé, le circuit n'apparaît que par ces deux propriétés. Toute question sur « tous les algorithmes à $T$ requêtes » devient une question sur « tous les polynômes bornés de degré $\le 2T$ », et c'est un théorème *pour tout polynôme* que PSI démontre.
2. Le rapport vaut $c\,d/\varepsilon^{9/2}$ : croissant en $d$ ($d \ge 1$) et décroissant en $\varepsilon$ ; or $\varepsilon \le \mathrm{Var} \le 1/4$ sur $[0,1]$, donc le pire cas **du domaine** est $\varepsilon = 1/4$, $d = 1$, où le rapport vaut $512\,c > 1$. Un seul point suffit parce que la monotonicité est établie — c'est l'argument de la cellule des constantes.
3. Majorité impaire sur $m$ bits : $s = (m+1)/2$ (au point à $(m-1)/2$ uns, chaque zéro flippe la valeur), $\deg = m$, $\max\mathrm{Inf} = \binom{2k}{k}/4^k \approx 1/\sqrt{\pi k}$ avec $k = (m-1)/2$, $\mathrm{Var} = 1$. Huang : $m \le \big((m+1)/2\big)^2$ ✓. AA : $1/\sqrt{\pi k} \ge c_{PM}/m^6$ ✓ très largement — la majorité est précisément la fonction qui montre que la dépendance en $d$ de la borne AA ne peut pas être meilleure que polynomiale.

</details>


## Pour aller plus loin

- **Carnets suivants** : [Complexity-07 — Complexité de Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) ; [Complexity-07c — Le bilan mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) — le même questionnement de provenance, appliqué à 372 preuves brutes.
- **Versant lac** : [sensitivity_lean](../../SymbolicAI/Lean/sensitivity_lean/README.md) (Huang, mêmes pins v4.33.0), [Lean-12 — Sensitivity Theorem](../../SymbolicAI/Lean/Lean-12-Sensitivity-Theorem.ipynb), [ANALYSE-06](../../SymbolicAI/Lean/Analyse/ANALYSE-06-Sensibilite-2026-Lean-Python.ipynb) et [ANALYSE-06b](../../SymbolicAI/Lean/Analyse/ANALYSE-06b-Sensibilite-2026-Lean.ipynb) pour l'audit Mathlib et la chaîne sensibilité.
- **Sources** : preprint PSI + dépôt `physical-superintelligence/aaronson-ambainis` (Apache-2.0, commit 2a6ceae) ; AA14, arXiv:0911.0996v3 — les deux archivés au gisement (chemins en tête de carnet). Plan de preuve voisin : Lovett–Zhang, *Fractional certificates for bounded functions* (ITCS 2023) — le dépôt PSI complète leur plan par l'inégalité toutes-paires qu'il laissait ouverte.


***

*Piste de provenance : ce carnet est né de la demande de distillation #25004 (2026-11-06) ; l'arbitrage « non-distillable sans port complet » suit la discipline des plis #19898 ; le dossier sensibilité↔degré↔influence continue dans le fil #21748. Les croisements correspondants vivent dans [ANALYSE-06](../../SymbolicAI/Lean/Analyse/ANALYSE-06-Sensibilite-2026-Lean-Python.ipynb) (§4) et le [README de `sensitivity_lean`](../../SymbolicAI/Lean/sensitivity_lean/README.md).*
